In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_percentage_error

In [ ]:
cat_features = ['region_name_cat', 'district_cat', 'corpus_cat', 'developer_cat',
                'hc_name_cat', 'interior_cat', 'class_cat', 'stage_cat']

group_cols = ['region_name_cat', 'district_cat']
target = 'price_target'
target_log = 'price_target_log'

In [ ]:
def clean_rooms(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().lower()
    if x == "студия":
        return 0.0
    if x in [">=4", ">= 4"]:
        return 4.0
    try:
        return float(x)
    except:
        return np.nan

In [ ]:
df = pd.read_csv("/content/period_1_train_data.csv")
test = pd.read_csv("/content/test_x.csv")

print(f"Train shape: {df.shape} | Test shape: {test.shape}")

In [ ]:
df['rooms_4'] = df['rooms_4'].apply(clean_rooms)
df['rooms_4'] = pd.to_numeric(df['rooms_4'], errors='coerce')   

df[target_log] = np.log1p(df[target])

df = df.replace([-999, -999.0], np.nan)

In [ ]:
df['rooms_4'] = df['rooms_4'].apply(clean_rooms)
df['rooms_4'] = pd.to_numeric(df['rooms_4'], errors='coerce')   

df[target_log] = np.log1p(df[target])

df = df.replace([-999, -999.0], np.nan)

In [ ]:
for col in cat_features:
    if col in df.columns:
        df[col] = df[col].fillna('missing').astype(str)     
        df[col] = df[col].astype('category')

In [ ]:
num_cols = [col for col in df.columns
            if col not in cat_features + ['agreement_date', target, target_log, 'id']
            and col in df.columns]

In [ ]:
group_medians = {}
global_medians = {}
for col in num_cols:
    group_medians[col] = df.groupby(group_cols)[col].median()
    global_medians[col] = df[col].median()

In [ ]:
for col in num_cols:
    df[col] = df.groupby(group_cols)[col].transform(lambda x: x.fillna(x.median()))
    df[col] = df[col].fillna(global_medians[col])

In [ ]:
df['agreement_date'] = pd.to_datetime(df['agreement_date'], errors='coerce')

df['year'] = df['agreement_date'].dt.year
df['month'] = df['agreement_date'].dt.month
df['dayofweek'] = df['agreement_date'].dt.dayofweek
df['quarter'] = df['agreement_date'].dt.quarter
df['age_days'] = (pd.Timestamp('2026-04-16') - df['agreement_date']).dt.days

df = df.drop(columns=['agreement_date'])

df['sq_per_room'] = df['square'] / df['rooms_4'].replace(0, 1)
df['floor_ratio'] = df['floor'] / (df.get('location_max_levels_max', pd.Series(1)).replace(0, 1) + 1)

cnt_cols = [col for col in num_cols if col.endswith('_cnt') and col in df.columns]
for col in cnt_cols:
    df[col] = np.log1p(df[col])

print(f"Train после предобработки: {df.shape}")

In [ ]:
X = df.drop(columns=[target, target_log, 'id'] if 'id' in df.columns else [target, target_log])
y = df[target_log]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42,
    stratify=df['region_name_cat'] if 'region_name_cat' in df.columns else None
)

cat_features_list = [col for col in cat_features if col in X_train.columns]

print("Категориальные признаки для модели:", cat_features_list)

print("\nCatBoost на валидации...")
train_pool = Pool(data=X_train, label=y_train, cat_features=cat_features_list)
val_pool   = Pool(data=X_val,   label=y_val,   cat_features=cat_features_list)

model = CatBoostRegressor(
    iterations=5000,
    learning_rate=0.08,
    depth=9,
    loss_function='RMSE',
    eval_metric='MAE',
    random_seed=42,
    early_stopping_rounds=200,
    task_type="CPU",          
    verbose=200,
    allow_writing_files=False
)

model.fit(train_pool, eval_set=val_pool)

In [ ]:
pred_log_val = model.predict(X_val)
pred_price_val = np.expm1(pred_log_val)
true_price_val = np.expm1(y_val.values)

mape_val = mean_absolute_percentage_error(true_price_val, pred_price_val) * 100

pred_log_train = model.predict(X_train)
pred_price_train = np.expm1(pred_log_train)
true_price_train = np.expm1(y_train.values)
mape_train = mean_absolute_percentage_error(true_price_train, pred_price_train) * 100

print(f"\nMAPE на VALIDATION: {mape_val:.4f}%")
print(f"MAPE на TRAIN:      {mape_train:.4f}%")

In [ ]:
print("\nфинальная модель на полном train...")
full_X = X
full_y = y
full_pool = Pool(full_X, full_y, cat_features=cat_features_list)

model_full = CatBoostRegressor(**model.get_params())
model_full.fit(full_pool, verbose=200)

In [ ]:
test['rooms_4'] = test['rooms_4'].apply(clean_rooms)
test['rooms_4'] = pd.to_numeric(test['rooms_4'], errors='coerce')
test = test.replace([-999, -999.0], np.nan)

for col in cat_features:
    if col in test.columns:
        test[col] = test[col].fillna('missing').astype(str).astype('category')

for col in num_cols:
    if col in test.columns:
        med_df = group_medians[col].reset_index().rename(columns={col: col + '_med'})
        test = test.merge(med_df, on=group_cols, how='left')
        test[col] = test[col].fillna(test[col + '_med'])
        test = test.drop(columns=[col + '_med'], errors='ignore')
        test[col] = test[col].fillna(global_medians[col])

test['agreement_date'] = pd.to_datetime(test['agreement_date'], errors='coerce')
test['year'] = test['agreement_date'].dt.year
test['month'] = test['agreement_date'].dt.month
test['dayofweek'] = test['agreement_date'].dt.dayofweek
test['quarter'] = test['agreement_date'].dt.quarter
test['age_days'] = (pd.Timestamp('2026-04-16') - test['agreement_date']).dt.days
test = test.drop(columns=['agreement_date'])

test['sq_per_room'] = test['square'] / test['rooms_4'].replace(0, 1)
test['floor_ratio'] = test['floor'] / (test.get('location_max_levels_max', pd.Series(1)).replace(0, 1) + 1)

for col in cnt_cols:
    if col in test.columns:
        test[col] = np.log1p(test[col])

In [ ]:
X_test = test.drop(columns=['id'] if 'id' in test.columns else [])
pred_log_test = model_full.predict(X_test)
pred_price_test = np.expm1(pred_log_test)

submission = pd.DataFrame({
    'id': test['id'],
    'price_target': pred_price_test
})

submission.to_csv('submission.csv', index=False)

print(f"Submission shape: {submission.shape}")
print(submission.head(10))